# File: 4_hello_torch.ipynb

Bu notebook PyTorch'a giriş ve temel uygulamaları içermektedir.
----------------------------------------------------------------

# PyTorch ile Makine Öğrenmesi Temelleri

Bu notebook, PyTorch kütüphanesinin temel kullanımını ve iki önemli makine öğrenmesi modelini uygulama adımlarını göstermektedir:

1. **Doğrusal Regresyon (Linear Regression)**: Sentetik bir veri seti üzerinde basit bir doğrusal model oluşturma
2. **Lojistik Regresyon (Logistic Regression)**: MNIST veri setindeki 3 ve 7 rakamlarını sınıflandıran ikili sınıflandırma modeli

Bu uygulamada PyTorch'un temel bileşenlerini (tensor işlemleri, modül oluşturma, kayıp fonksiyonları, optimizasyon) ve modellerin eğitim ve test süreçlerini inceleyeceğiz.

## Gerekli Kütüphanelerin İçe Aktarılması

Aşağıdaki kütüphaneleri kullanacağız:

- **PyTorch**: Derin öğrenme çerçevesi (torch, torch.nn, torch.optim)
- **NumPy**: Sayısal hesaplamalar için
- **Matplotlib**: Veri görselleştirme
- **Scikit-learn**: Veri seti oluşturma, bölme ve metrik hesaplama
- **idx2numpy**: MNIST verilerini okuma

In [ ]:
import torch
import torch.nn as nn
from torch.optim import SGD

import numpy as np
import matplotlib.pyplot as plt

from sklearn import datasets
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

import idx2numpy

## Doğrusal Regresyon için Veri Seti Hazırlama

Aşağıda, doğrusal regresyon modelimizi eğitmek ve test etmek için yapay bir veri seti oluşturuyoruz. Scikit-learn kütüphanesinin `make_regression` fonksiyonunu kullanarak:

- 500 örnek (num_samples)
- 2 özellik (num_features)
- Gürültü seviyesi 5 (noise)
- Tutarlılık için rastgele durum (random_state) 42

Oluşturulan hedef değişkeni (y_np), ikinci boyut eklemek için yeniden şekillendiriyoruz (expand_dims), bu PyTorch tarafından beklenen formattır.

In [ ]:
num_samples = 500
num_features = 2
X_np, y_np = datasets.make_regression(num_samples, num_features, noise=5, random_state=42)
y_np = np.expand_dims(y_np, axis=1)

In [ ]:
print("X shape:", X_np.shape)
print("İlk 5 X:")
print(X_np[:5])
print("-" * 40)
print("y shape:", y_np.shape)
print("ilk 5 y:")
print(y_np[:5])

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X_np, y_np, test_size=0.2, random_state=42)

## NumPy Dizilerinden PyTorch Tensörlerine Dönüştürme

PyTorch ile işlem yapabilmek için NumPy dizilerini PyTorch tensörlerine dönüştürmemiz gerekiyor. Bu dönüşüm sırasında veri tipini `float32` olarak belirtiyoruz çünkü PyTorch modelleri genellikle float32 veri tipiyle daha verimli çalışır.

Veri setimizi eğitim (%80) ve test (%20) olarak ikiye ayırdıktan sonra, her bir kısmı PyTorch tensörlerine dönüştürüyoruz.

In [ ]:
X_train = torch.tensor(X_train, dtype=torch.float32)
y_train = torch.tensor(y_train, dtype=torch.float32)

X_test = torch.tensor(X_test, dtype=torch.float32)
y_test = torch.tensor(y_test, dtype=torch.float32)

In [ ]:
print(X_train.shape)
print(y_train.shape)
print(X_test.shape)
print(y_test.shape)

## Doğrusal Regresyon Modeli Oluşturma

PyTorch'un `nn.Linear` modülünü kullanarak basit bir doğrusal regresyon modeli oluşturuyoruz. Bu modül, matematiksel olarak $y = wx + b$ formülünü uygular, burada:

- `num_features` girdi boyutu (özellik sayısı)
- `1` çıktı boyutu (tahmin edilen değer)

Ayrıca, model eğitimi için gerekli bileşenleri tanımlıyoruz:

- **Kayıp Fonksiyonu**: Ortalama Kare Hata (MSE), $\frac{1}{n}\sum_{i=1}^{n}(y_i - \hat{y}_i)^2$
- **Optimizer**: Stokastik Gradyan İnişi (SGD), öğrenme hızı (learning rate) 0.001

In [ ]:
lineer_model = nn.Linear(num_features, 1)
loss_func = nn.MSELoss()
optimizer = SGD(lineer_model.parameters(), lr=0.001)

In [ ]:
for name, param in lineer_model.named_parameters():
    if param.requires_grad:
        print(name, param.data)

## Model Eğitimi

Doğrusal regresyon modelimizi eğitim verisi üzerinde 2000 epoch boyunca eğitiyoruz. Her bir epoch'ta şu adımlar gerçekleşir:

1. **İleri Yayılım (Forward Pass)**: Model girdileri alır ve tahminleri üretir (`y_hat`)
2. **Kayıp Hesaplaması**: Tahminler ile gerçek değerler arasındaki hata hesaplanır
3. **Geri Yayılım (Backward Pass)**: Kayıp değerine göre gradyanlar hesaplanır (`loss.backward()`)
4. **Parametre Güncelleme**: Optimizer, gradyanlara göre model parametrelerini günceller
5. **Gradyanların Sıfırlanması**: Bir sonraki iterasyon için gradyanlar sıfırlanır

Her 20 epoch'ta bir, eğitim sürecini izlemek için kayıp değeri kaydedilir.

In [ ]:
history = {"epoch": [], "loss": []}

for epoch in range(2000):
    y_hat = lineer_model(X_train)
    loss = loss_func(y_hat, y_train)
    loss.backward()
    optimizer.step()
    optimizer.zero_grad()

    if (epoch + 1) % 20 == 0:
        history["epoch"].append(epoch)
        history["loss"].append(loss.item())

## Eğitim Sürecinin Görselleştirilmesi

Doğrusal regresyon modelinin eğitim sürecini görselleştirmek için, epoch'lara karşı kayıp değerlerini çiziyoruz. Bu grafik, modelin eğitim sırasında nasıl geliştiğini gösterir. Kayıp değerinin zamanla azalması, modelin veriyi daha iyi temsil etmeye başladığını gösterir.

In [ ]:
plt.title("Lin Reg Model Loss")
plt.plot(history["epoch"], history["loss"], linewidth=3)
plt.xlabel("Epoch")
plt.ylabel("MSE")
plt.show()

In [ ]:
for name, param in lineer_model.named_parameters():
    if param.requires_grad:
        print(name, param.data)

## Test Veri Seti Üzerinde Model Performansı

Eğitilmiş modelimizi test veri seti üzerinde değerlendiriyoruz. Bu, modelin daha önce görmediği veriler üzerindeki genelleme yeteneğini kontrol etmemizi sağlar. Grafikte:

- Kırmızı çizgi: Gerçek değerler (y_test)
- Mavi çizgi: Model tahminleri (y_predicted)

İki çizginin ne kadar yakın olduğu, modelin tahmin performansını gösterir.

In [ ]:
y_predicted = lineer_model(X_test).detach().numpy()
plt.figure(figsize=(12, 4))
plt.title("Lin Reg Testleri")
plt.plot(np.linspace(0, 10, 100), y_test.numpy(), label="real", c="r")
plt.plot(np.linspace(0, 10, 100), y_predicted, label="prediction", c="b")
plt.legend()
plt.show()

# Lojistik Regresyon ile MNIST Veri Seti Sınıflandırma

Şimdi daha karmaşık bir probleme geçiyoruz: El yazısı rakamları sınıflandırma. MNIST veri setinden 3 ve 7 rakamlarını alıp ikili bir sınıflandırma problemi olarak modelleyeceğiz.

## 1. Adım: Veri Setini Okuma ve Hazırlama

- MNIST veri setini yüklüyoruz
- Görüntüleri düzleştiriyoruz (784 boyutlu vektörlere)
- Piksel değerlerini 0-1 aralığına normalize ediyoruz
- Sadece 3 (etiket 0) ve 7 (etiket 1) rakamlarını seçiyoruz

In [ ]:
MNIST_DIR = "mnist/"
train_arr = idx2numpy.convert_from_file(MNIST_DIR + "train-images-idx3-ubyte")
train_labels = idx2numpy.convert_from_file(MNIST_DIR + "train-labels-idx1-ubyte")

X_train = train_arr.reshape(60000, -1)
X_train = X_train / 255.0
y_train = np.copy(train_labels)

X_3 = X_train[y_train == 3]
y_3 = np.zeros(X_3.shape[0])

X_7 = X_train[y_train == 7]
y_7 = np.ones(X_7.shape[0])

X_3_7 = np.append(X_3, X_7, axis=0)
y_3_7 = np.append(y_3, y_7)

ds_check_indexes = [0, 1000, 5000, 5200, 6200, 11000, 12300, 12301, 12395]

plt.figure(figsize=(9, 9))

for i, index in enumerate(ds_check_indexes):
    plt.subplot(3, 3, i+1)
    plt.title(str(y_3_7[index]))
    plt.imshow(X_3_7[index].reshape(28, 28), cmap="gray")

plt.tight_layout()
plt.show()

In [ ]:
y_3_7 = np.expand_dims(y_3_7, axis=1)
X_train, X_test, y_train, y_test = train_test_split(X_3_7, y_3_7, test_size=0.2, random_state=42)

print(X_train.shape, y_train.shape)
print(X_test.shape, y_test.shape)

## NumPy Dizilerini PyTorch Tensörlerine Dönüştürme

Doğrusal regresyon örneğinde olduğu gibi, verilerimizi PyTorch tensörlerine dönüştürüyoruz. Bu, PyTorch'un otomatik türev alma mekanizması ve GPU hızlandırması gibi özelliklerinden yararlanmamızı sağlar.

In [ ]:
X_train = torch.tensor(X_train, dtype=torch.float32)
y_train = torch.tensor(y_train, dtype=torch.float32)

X_test = torch.tensor(X_test, dtype=torch.float32)
y_test = torch.tensor(y_test, dtype=torch.float32)

## Lojistik Regresyon Modeli Oluşturma

İkili sınıflandırma için özel bir lojistik regresyon modeli tanımlıyoruz. Bu modelde:

1. `__init__` metodu: Giriş özelliklerinin sayısını alır ve bir doğrusal katman oluşturur
2. `forward` metodu: Giriş verilerini alır, doğrusal dönüşüm uygular ve sigmoid aktivasyonu ile 0-1 arasında bir olasılık değeri döndürür

Sigmoid fonksiyonu $\sigma(z) = \frac{1}{1 + e^{-z}}$ formülü ile hesaplanır ve herhangi bir sayıyı 0-1 aralığına sıkıştırır.

In [ ]:
class LogReg(nn.Module):
    def __init__(self, n_input_features):
        super().__init__()
        self.linear = nn.Linear(n_input_features, 1)

    def forward(self, x):
        y_lin = self.linear(x)
        y_hat = torch.sigmoid(y_lin)
        return y_hat

In [ ]:
print(X_train.shape)
print(X_train.shape[1])

## Lojistik Regresyon Modelinin Eğitimi

Lojistik regresyon modelimizi eğitim veri seti üzerinde 1000 epoch boyunca eğitiyoruz. Doğrusal regresyondan farklı olarak:

- Kayıp fonksiyonu olarak Binary Cross Entropy (BCE) kullanıyoruz, bu ikili sınıflandırma problemleri için uygundur
- Öğrenme hızı daha yüksek (0.005), bu veri seti için daha hızlı öğrenmeye olanak tanır

Eğitim süreci, doğrusal regresyondaki ile aynı adımları takip eder: ileri yayılım, kayıp hesaplama, geri yayılım, parametre güncelleme ve gradyan sıfırlama.

In [ ]:
num_features = X_train.shape[1]
logistic_model = LogReg(num_features)
loss_func = nn.BCELoss()
optimizer = SGD(logistic_model.parameters(), lr=0.005)

history = {"epoch": [], "loss": []}
for epoch in range(1000):
    y_hat = logistic_model(X_train)
    loss = loss_func(y_hat, y_train)
    loss.backward()
    optimizer.step()
    optimizer.zero_grad()

    if (epoch + 1) % 20 == 0:
        history["epoch"].append(epoch)
        history["loss"].append(loss.item())

## Eğitim Sürecinin Görselleştirilmesi

Lojistik regresyon modelinin eğitim sürecini görselleştiriyoruz. BCE kayıp değeri zamanla azalırsa, model sınıflandırma görevinde giderek daha başarılı hale geliyor demektir.

In [ ]:
plt.title("Logistic Reg Model Loss")
plt.plot(history["epoch"], history["loss"], linewidth=3)
plt.xlabel("Epoch")
plt.ylabel("BCE Loss")
plt.show()

In [ ]:
for name, param in logistic_model.named_parameters():
    if param.requires_grad:
        print(name, param.shape, param.data)

## Model Performansının Değerlendirilmesi

Eğitilmiş lojistik regresyon modelimizi test veri seti üzerinde değerlendiriyoruz:

1. Test verileri üzerinde tahminler yapıyoruz (`with torch.no_grad()` bloğu, gradyanların hesaplanmasını devre dışı bırakarak tahmin sürecini hızlandırır)
2. Olasılık değerlerini (0-1 arası) ikili sınıf etiketlerine dönüştürüyoruz (eşik değeri 0.5)
3. Accuracy (doğruluk) metriğini hesaplıyoruz, bu, doğru tahmin edilen örneklerin toplam örnek sayısına oranıdır

Bu değerlendirme, modelimizin daha önce görmediği 3 ve 7 rakamlarını ne kadar iyi ayırt edebildiğini gösterir.

In [ ]:
with torch.no_grad():
    y_predicted = logistic_model(X_test).numpy()

In [ ]:
y_predicted[:3]

In [ ]:
print(y_predicted.shape)

In [ ]:
y_predicted = np.squeeze(y_predicted)
print(y_predicted.shape)
y_predicted[:3]

In [ ]:
y_test = np.squeeze(y_test.numpy())
y_test[:3]

In [ ]:
y_predicted_classes = np.where(y_predicted > 0.5, 1, 0)
y_predicted_classes[:3]

In [ ]:
acc = accuracy_score(y_test, y_predicted_classes)
print(acc)